# bigcompute.science Research Agent

Run the autonomous research agent from Google Colab. The agent monitors GPU experiments, harvests results, analyzes data, runs multi-model peer reviews, fixes issues, and deploys updates.

**You need ONE of these:**
- An OpenAI API key (`OPENAI_API_KEY`)
- An Anthropic API key (`ANTHROPIC_API_KEY`)
- Both (for multi-model reviews — recommended)

Set your key(s) in Colab Secrets (🔑 icon in the sidebar) or paste below.

> **Part of [bigcompute.science](https://bigcompute.science) — open computational mathematics.**

In [ ]:
# === Step 1: Clone the repo ===
!git clone https://github.com/cahlen/idontknow.git 2>/dev/null || (cd idontknow && git pull)
%cd idontknow
!pip install -q httpx

In [ ]:
# === Step 2: Set your API key(s) ===
# Option A: Use Colab Secrets (recommended — click 🔑 in sidebar)
import os
try:
    from google.colab import userdata
    if userdata.get('OPENAI_API_KEY', ''):
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        print('✓ OpenAI key loaded from Colab Secrets')
    if userdata.get('ANTHROPIC_API_KEY', ''):
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
        print('✓ Anthropic key loaded from Colab Secrets')
except:
    pass

# Option B: Paste directly (less secure, but works)
# os.environ['OPENAI_API_KEY'] = 'sk-proj-...'     # OpenAI
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'   # Anthropic

has_openai = bool(os.environ.get('OPENAI_API_KEY'))
has_anthropic = bool(os.environ.get('ANTHROPIC_API_KEY'))
print(f'\nAvailable: OpenAI={has_openai}, Anthropic={has_anthropic}')
if not has_openai and not has_anthropic:
    print('⚠️  No API key set — set one above to continue')

In [ ]:
# === Step 3: Run the agent (dry run — see what it would do) ===
!python3 scripts/research_agent.py --once --dry-run

In [ ]:
# === Step 4: Run for real (one tick) ===
# This will harvest results, analyze them, and report findings.
# It will NOT commit/push (you'd need git auth for that).
!python3 scripts/research_agent.py --once --phase harvest

In [ ]:
# === Step 5: Run a peer review on a specific finding ===
# This uses your API key to get a fresh review from a different model.
finding = 'zaremba-density-phase-transition'  # @param {type:"string"}
model = 'gpt-4.1'  # @param ["gpt-4.1", "gpt-4.1-mini", "o3", "o3-pro"]

os.environ['API_KEY'] = os.environ.get('OPENAI_API_KEY', os.environ.get('ANTHROPIC_API_KEY', ''))
!python3 scripts/reviews/run_review.py --slug {finding} --model {model} --provider openai --skip-existing

In [ ]:
# === Step 6: View all reviews for a finding ===
import json, glob

finding = 'zaremba-density-phase-transition'  # @param {type:"string"}

for f in sorted(glob.glob(f'docs/verifications/{finding}*review*.json')):
    with open(f) as fh:
        r = json.load(fh)
    model = r.get('reviewer', {}).get('model', '?')
    verdict = r.get('overall_verdict', '?')
    cert = r.get('certification_recommendation', '?')
    print(f'{model:20s} {verdict:25s} {cert}')

In [ ]:
# === Step 7: Rebuild the manifest (aggregates all reviews) ===
!python3 scripts/reviews/aggregate.py
!python3 scripts/reviews/validate.py --all

## What's Next?

**To contribute your review back:**
1. Download the review JSON from `docs/verifications/`
2. Open a PR to [cahlen/idontknow](https://github.com/cahlen/idontknow)
3. Your review will be added to the audit ledger

**To run the full autonomous loop** (needs a machine with GPUs):
```bash
export OPENAI_API_KEY='sk-...'  # or ANTHROPIC_API_KEY
nohup ./scripts/run_agent.sh --loop 10m &
```

**More info:**
- [AGENTS.md](https://github.com/cahlen/idontknow/blob/main/AGENTS.md) — full contribution guide
- [Verification page](https://bigcompute.science/verification/) — all reviews and certifications
- [MCP Server](https://mcp.bigcompute.science/mcp) — 23 tools, no auth